           AutoStream AI Agent - Complete Consolidated Codebase          
                                                                            
  This file contains all the code for the production-ready conversational   
  AI agent. For easier navigation, code is organized by section.            
                                                                            
  Sections:                                                                 
    1. Knowledge Base (RAG System)                                          
    2. Tools & Validation                                                   
    3. Personalization Engine                                               
    4. Analytics Engine                                                     
    5. A/B Testing Framework                                                
    6. Main Agent (LangGraph)                                               
                               

#### SECTION 1: KNOWLEDGE BASE (RAG SYSTEM)

In [1]:
import json
from typing import List, Dict, Any, Optional, Tuple
from dataclasses import dataclass, field, asdict
from datetime import datetime
from enum import Enum
import re

In [2]:
class KnowledgeBase:
    """Simple RAG implementation using keyword matching and semantic search"""
    
    def __init__(self, kb_path: str = "knowledge_base.json"):
        """Initialize knowledge base from JSON file"""
        try:
            with open(kb_path, 'r') as f:
                self.data = json.load(f)
        except FileNotFoundError:
            # Use default knowledge base if file not found
            self.data = self._get_default_knowledge_base()
    
    def _get_default_knowledge_base(self) -> Dict:
        """Return default knowledge base"""
        return {
            "company": {
                "name": "AutoStream",
                "description": "AI-powered automated video editing platform for content creators",
                "website": "https://autostream.ai"
            },
            "pricing": {
                "basic_plan": {
                    "name": "Basic Plan",
                    "price": "$29/month",
                    "videos_per_month": 10,
                    "resolution": "720p",
                    "ai_captions": False,
                    "support": "Community support",
                    "features": [
                        "10 videos per month",
                        "720p resolution",
                        "Basic templates",
                        "Auto-cut silence"
                    ]
                },
                "pro_plan": {
                    "name": "Pro Plan",
                    "price": "$79/month",
                    "videos_per_month": "Unlimited",
                    "resolution": "4K",
                    "ai_captions": True,
                    "support": "24/7 priority support",
                    "features": [
                        "Unlimited videos",
                        "4K resolution",
                        "AI captions",
                        "Advanced templates",
                        "Custom branding",
                        "24/7 support",
                        "API access"
                    ]
                }
            },
            "policies": {
                "refund_policy": "No refunds after 7 days of purchase",
                "support_availability": {
                    "basic": "Community support only",
                    "pro": "24/7 priority support"
                },
                "trial": "14-day free trial available for both plans"
            },
            "platforms_supported": [
                "YouTube", "Instagram", "TikTok", "LinkedIn", "Twitch", "Facebook"
            ]
        }
    
    def retrieve(self, query: str) -> Dict[str, Any]:
        """
        Retrieve relevant information from knowledge base based on query.
        Returns context and confidence score.
        """
        query_lower = query.lower()
        context = []
        
        # Pricing queries
        if any(word in query_lower for word in ["price", "pricing", "cost", "plan", "paid", "pay"]):
            context.append(self._format_pricing())
            return {
                "type": "pricing_query",
                "context": context,
                "confidence": 0.95
            }

In [7]:
def classify_query(self, query):
    query_lower = query.lower()
    context = []

    # Pricing queries
    if any(word in query_lower for word in ["price", "pricing", "cost", "plan", "paid", "pay"]):
        context.append(self._format_pricing())
        return {
            "type": "pricing_query",
            "context": context,
            "confidence": 0.95
        }

    # Features queries
    if any(word in query_lower for word in ["features", "does it", "can", "support", "capability", "do you"]):
        context.append(self._format_features())
        return {
            "type": "features_query",
            "context": context,
            "confidence": 0.9
        }

    # Support queries
    if any(word in query_lower for word in ["support", "help", "customer service"]):
        context.append(self._format_support())
        return {
            "type": "support_query",
            "context": context,
            "confidence": 0.9
        }

    # Refund/Policy queries
    if any(word in query_lower for word in ["refund", "money back", "return", "policy"]):
        context.append(self._format_policies())
        return {
            "type": "policy_query",
            "context": context,
            "confidence": 0.9
        }

    # Trial queries
    if any(word in query_lower for word in ["free", "trial", "test"]):
        context.append(self._format_trial())
        return {
            "type": "trial_query",
            "context": context,
            "confidence": 0.85
        }

    # Default
    context.append(self._format_general_info())
    return {
        "type": "general_query",
        "context": context,
        "confidence": 0.7
    }

In [10]:
    def _format_pricing(self) -> str:
        """Format pricing information"""
        pricing = self.data["pricing"]
        return f"""
**AutoStream Pricing:**

{pricing['basic_plan']['name']} - {pricing['basic_plan']['price']}
- {pricing['basic_plan']['videos_per_month']} videos/month
- {pricing['basic_plan']['resolution']} resolution
- {', '.join(pricing['basic_plan']['features'])}

{pricing['pro_plan']['name']} - {pricing['pro_plan']['price']}
- Unlimited videos
- {pricing['pro_plan']['resolution']} resolution
- {', '.join(pricing['pro_plan']['features'])}
"""
    
    def _format_features(self) -> str:
        """Format features information"""
        pricing = self.data["pricing"]
        return f"""
**AutoStream Features:**

Basic Plan includes: {', '.join(pricing['basic_plan']['features'])}

Pro Plan includes: {', '.join(pricing['pro_plan']['features'])}

Works with: {', '.join(self.data['platforms_supported'])}
"""
    
    def _format_support(self) -> str:
        """Format support information"""
        policies = self.data["policies"]
        return f"""
**Support Options:**
- Basic Plan: {policies['support_availability']['basic']}
- Pro Plan: {policies['support_availability']['pro']}
"""
    
    def _format_policies(self) -> str:
        """Format policy information"""
        policies = self.data["policies"]
        return f"""
**Company Policies:**
- Refund Policy: {policies['refund_policy']}
- Trial: {policies['trial']}
"""
    
    def _format_trial(self) -> str:
        """Format trial information"""
        return "AutoStream offers a 14-day free trial for both Basic and Pro plans. No credit card required!"
    
    def _format_general_info(self) -> str:
        """Format general information"""
        return f"""
**About AutoStream:**
{self.data['company']['description']}

AutoStream helps content creators automate video editing with AI-powered tools.
Get started with our free 14-day trial today!
"""

#### SECTION 2: TOOLS & VALIDATION

In [12]:
@dataclass
class LeadData:
    """Structure to hold lead information"""
    name: Optional[str] = None
    email: Optional[str] = None
    platform: Optional[str] = None
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())
    
    def is_complete(self) -> bool:
        """Check if all required fields are filled"""
        return all([self.name, self.email, self.platform])
    
    def to_dict(self) -> dict:
        """Convert to dictionary"""
        return asdict(self)


def mock_lead_capture(name: str, email: str, platform: str) -> dict:
    """Mock API function to capture leads"""
    print(f"✓ Lead captured successfully: {name}, {email}, {platform}")
    
    lead = LeadData(name=name, email=email, platform=platform)
    
    return {
        "success": True,
        "message": f"Welcome to AutoStream, {name}! We'll contact you shortly at {email}.",
        "lead_id": f"LEAD_{datetime.now().strftime('%Y%m%d%H%M%S')}",
        "data": lead.to_dict()
    }


class LeadQualifier:
    """Determine if user is a high-intent lead"""
    
    high_intent_keywords = [
        "sign up", "register", "buy", "purchase", "subscribe",
        "get started", "start using", "want to try", "interested",
        "how much", "pricing", "plans", "my channel", "my videos"
    ]
    
    @staticmethod
    def is_high_intent(message: str) -> bool:
        """Check if message indicates high-intent to convert"""
        message_lower = message.lower()
        
        # Check for explicit high-intent keywords
        for keyword in LeadQualifier.high_intent_keywords:
            if keyword in message_lower:
                return True
        
        # Check for platform mentions
        platforms = ["youtube", "instagram", "tiktok", "linkedin", "twitch", "facebook"]
        platform_count = sum(1 for p in platforms if p in message_lower)
        
        if platform_count > 0 and any(w in message_lower for w in ["channel", "account", "my", "i have"]):
            return True
        
        return False


def validate_email(email: str) -> bool:
    """Simple email validation"""
    pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
    return re.match(pattern, email) is not None


def validate_name(name: str) -> bool:
    """Validate name is not empty and has reasonable length"""
    return len(name.strip()) >= 2 and len(name) <= 100


def validate_platform(platform: str) -> bool:
    """Validate platform is one of supported platforms"""
    supported = ["youtube", "instagram", "tiktok", "linkedin", "twitch", "facebook"]
    return platform.lower() in supported

#### SECTION 3: PERSONALIZATION ENGINE

In [13]:
class ConversationStyle(Enum):
    """Different conversation styles"""
    CASUAL = "casual"
    PROFESSIONAL = "professional"
    ENTHUSIASTIC = "enthusiastic"
    EDUCATIONAL = "educational"
    CONSULTATIVE = "consultative"


class UserSegment(Enum):
    """User segments for personalization"""
    BEGINNER = "beginner"
    INTERMEDIATE = "intermediate"
    ADVANCED = "advanced"
    ENTERPRISE = "enterprise"


@dataclass
class PersonalizationProfile:
    """User personalization profile"""
    user_id: str
    segment: UserSegment = UserSegment.BEGINNER
    conversation_style: ConversationStyle = ConversationStyle.CASUAL
    preferred_language: str = "English"
    interaction_count: int = 0
    last_intent: Optional[str] = None
    platform_interest: Optional[str] = None


class ConversationPatterns:
    """Repository of advanced conversation patterns"""
    
    GREETING_RESPONSES = {
        ConversationStyle.CASUAL: [
            "Hey there! 👋 Welcome to AutoStream!",
            "Hey! What's up? How can I help you today?",
            "Welcome! Great to see you here!",
        ],
        ConversationStyle.PROFESSIONAL: [
            "Good day. How can I assist you with AutoStream?",
            "Welcome. How may I help you today?",
            "Hello. What would you like to know about our service?",
        ],
        ConversationStyle.ENTHUSIASTIC: [
            "Welcome to AutoStream! 🎬 Ready to revolutionize your video editing?",
            "Hey! So excited to show you what we can do!",
            "Welcome! Let's find the perfect solution for you!",
        ],
        ConversationStyle.EDUCATIONAL: [
            "Welcome! I'm here to help you understand AutoStream's capabilities.",
            "Hello! I'd love to guide you through our features.",
            "Hi! Let's explore how AutoStream can help you.",
        ],
    }
    
    PRICING_RESPONSES = {
        ConversationStyle.CASUAL: {
            "basic": "Our Basic Plan is $29/month - perfect for getting started with 10 videos a month in 720p.",
            "pro": "The Pro Plan is $79/month and gives you unlimited videos in 4K plus AI captions!",
            "comparison": "Basic is $29 (10 videos/month, 720p), Pro is $79 (unlimited videos, 4K). Pro also includes 24/7 support."
        },
        ConversationStyle.PROFESSIONAL: {
            "basic": "The Basic Plan costs $29/month and includes 10 monthly videos with 720p resolution.",
            "pro": "The Pro Plan is priced at $79/month, offering unlimited videos in 4K with AI captions.",
            "comparison": "Basic: $29/month with 10 monthly videos and 720p. Pro: $79/month with unlimited videos and 4K."
        },
        ConversationStyle.ENTHUSIASTIC: {
            "basic": "Just $29/month to get started! 10 videos a month at 720p - awesome value! 🎉",
            "pro": "Only $79/month for UNLIMITED videos in stunning 4K! Plus AI captions! Incredible deal! 🚀",
            "comparison": "Basic is just $29 (10 vids, 720p), but the Pro at $79 is AMAZING - unlimited 4K! 🎥"
        }
    }
    
    PLATFORM_ADVICE = {
        "youtube": {
            "casual": "YouTube creators love our 4K support - your viewers will be blown away!",
            "professional": "For YouTube creators, our 4K capability and unlimited video uploads are particularly valuable.",
            "enthusiastic": "Oh man, YouTubers LOVE AutoStream! Your channel is about to level up! 🚀"
        },
        "instagram": {
            "casual": "Instagram's all about quick, slick edits. Our Pro plan is perfect!",
            "professional": "For Instagram content, rapid iteration and AI captions are key advantages.",
            "enthusiastic": "Instagram creators are killing it with AutoStream! Your Reels are about to go viral! 📱"
        },
        "tiktok": {
            "casual": "TikTok creators swear by our speed. You can edit like lightning!",
            "professional": "TikTok requires fast turnaround - our unlimited video limit enables high-volume creation.",
            "enthusiastic": "TikTok? AutoStream is MADE for TikTok creators! Trending videos incoming! 🎬"
        }
    }
    
    @staticmethod
    def get_greeting(style: ConversationStyle) -> str:
        """Get greeting for style"""
        import random
        responses = ConversationPatterns.GREETING_RESPONSES.get(
            style,
            ConversationPatterns.GREETING_RESPONSES[ConversationStyle.CASUAL]
        )
        return random.choice(responses)
    
    @staticmethod
    def get_pricing_info(style: ConversationStyle, plan_type: str) -> str:
        """Get pricing info for style"""
        responses = ConversationPatterns.PRICING_RESPONSES.get(
            style,
            ConversationPatterns.PRICING_RESPONSES[ConversationStyle.CASUAL]
        )
        return responses.get(plan_type, "")


class PersonalizationEngine:
    """Engine for personalizing conversations based on user profile"""
    
    def __init__(self):
        self.profiles: Dict[str, PersonalizationProfile] = {}
    
    def create_profile(self, user_id: str) -> PersonalizationProfile:
        """Create user profile"""
        profile = PersonalizationProfile(user_id=user_id)
        self.profiles[user_id] = profile
        return profile
    
    def get_profile(self, user_id: str) -> Optional[PersonalizationProfile]:
        """Get user profile"""
        return self.profiles.get(user_id)
    
    def recommend_style(self, profile: PersonalizationProfile) -> ConversationStyle:
        """Recommend conversation style based on profile"""
        if profile.segment == UserSegment.BEGINNER:
            return ConversationStyle.ENTHUSIASTIC
        elif profile.segment == UserSegment.INTERMEDIATE:
            return ConversationStyle.PROFESSIONAL
        else:
            return ConversationStyle.EDUCATIONAL
    
    def update_profile(self, user_id: str, **kwargs):
        """Update user profile"""
        if user_id in self.profiles:
            profile = self.profiles[user_id]
            for key, value in kwargs.items():
                if hasattr(profile, key):
                    setattr(profile, key, value)
    
    def increment_interaction(self, user_id: str):
        """Increment interaction count"""
        if user_id in self.profiles:
            self.profiles[user_id].interaction_count += 1


#### SECTION 4: ANALYTICS ENGINE

In [15]:
class IntentType(Enum):
    """Enum for intent types"""
    GREETING = "greeting"
    INQUIRY = "inquiry"
    HIGH_INTENT = "high_intent"


class ConversionStatus(Enum):
    """Enum for lead conversion status"""
    INITIATED = "initiated"
    IN_PROGRESS = "in_progress"
    COMPLETED = "completed"
    ABANDONED = "abandoned"


@dataclass
class ConversationMetrics:
    """Track metrics for a single conversation"""
    conversation_id: str
    start_time: datetime = field(default_factory=datetime.now)
    end_time: datetime = None
    user_id: str = None
    
    intents_detected: List[str] = field(default_factory=list)
    intent_confidence_scores: List[float] = field(default_factory=list)
    
    total_turns: int = 0
    user_messages: int = 0
    agent_responses: int = 0
    
    conversion_status: ConversionStatus = ConversionStatus.INITIATED
    lead_data_collected: Dict[str, bool] = field(default_factory=lambda: {
        "name": False,
        "email": False,
        "platform": False
    })
    
    average_response_time: float = 0.0
    total_tokens_used: int = 0
    rag_queries: int = 0
    successful_captures: int = 0
    failed_captures: int = 0
    
    errors: List[str] = field(default_factory=list)
    
    def to_dict(self) -> dict:
        """Convert to dictionary for storage"""
        return {
            "conversation_id": self.conversation_id,
            "start_time": self.start_time.isoformat(),
            "end_time": self.end_time.isoformat() if self.end_time else None,
            "user_id": self.user_id,
            "intents_detected": self.intents_detected,
            "total_turns": self.total_turns,
            "conversion_status": self.conversion_status.value,
            "lead_data_collected": self.lead_data_collected,
            "average_response_time": self.average_response_time,
            "total_tokens_used": self.total_tokens_used,
            "rag_queries": self.rag_queries,
        }


class AnalyticsEngine:
    """Central analytics engine for tracking agent performance"""
    
    def __init__(self):
        self.conversations: Dict[str, ConversationMetrics] = {}
        self.aggregated_metrics = {
            "total_conversations": 0,
            "total_leads_captured": 0,
            "conversion_rate": 0.0,
            "average_turns_per_conversation": 0,
            "intent_distribution": {},
            "common_errors": {},
            "average_response_time": 0.0
        }
    
    def start_conversation(self, conversation_id: str, user_id: str = None) -> ConversationMetrics:
        """Start tracking a new conversation"""
        metrics = ConversationMetrics(conversation_id=conversation_id, user_id=user_id)
        self.conversations[conversation_id] = metrics
        return metrics
    
    def end_conversation(self, conversation_id: str):
        """Mark conversation as ended"""
        if conversation_id in self.conversations:
            self.conversations[conversation_id].end_time = datetime.now()
            self._update_aggregates()
    
    def record_intent(self, conversation_id: str, intent: str, confidence: float = 0.9):
        """Record detected intent"""
        if conversation_id in self.conversations:
            metrics = self.conversations[conversation_id]
            metrics.intents_detected.append(intent)
            metrics.intent_confidence_scores.append(confidence)
    
    def record_turn(self, conversation_id: str, is_user_message: bool):
        """Record a conversation turn"""
        if conversation_id in self.conversations:
            metrics = self.conversations[conversation_id]
            metrics.total_turns += 1
            if is_user_message:
                metrics.user_messages += 1
            else:
                metrics.agent_responses += 1
    
    def record_lead_capture(self, conversation_id: str, success: bool):
        """Record lead capture attempt"""
        if conversation_id in self.conversations:
            metrics = self.conversations[conversation_id]
            if success:
                metrics.successful_captures += 1
                metrics.conversion_status = ConversionStatus.COMPLETED
            else:
                metrics.failed_captures += 1
    
    def get_conversion_funnel(self) -> Dict[str, int]:
        """Get conversion funnel breakdown"""
        funnel = {
            "total_conversations": 0,
            "with_greeting": 0,
            "with_inquiry": 0,
            "high_intent_detected": 0,
            "name_collected": 0,
            "email_collected": 0,
            "platform_collected": 0,
            "leads_captured": 0
        }
        
        for metrics in self.conversations.values():
            funnel["total_conversations"] += 1
            if "greeting" in metrics.intents_detected:
                funnel["with_greeting"] += 1
            if "inquiry" in metrics.intents_detected:
                funnel["with_inquiry"] += 1
            if "high_intent" in metrics.intents_detected:
                funnel["high_intent_detected"] += 1
            if metrics.lead_data_collected.get("name"):
                funnel["name_collected"] += 1
            if metrics.lead_data_collected.get("email"):
                funnel["email_collected"] += 1
            if metrics.lead_data_collected.get("platform"):
                funnel["platform_collected"] += 1
            if metrics.conversion_status == ConversionStatus.COMPLETED:
                funnel["leads_captured"] += 1
        
        return funnel
    
    def _update_aggregates(self):
        """Update aggregated metrics"""
        if not self.conversations:
            return
        
        total_convs = len(self.conversations)
        successful_leads = sum(
            1 for m in self.conversations.values()
            if m.conversion_status == ConversionStatus.COMPLETED
        )
        
        self.aggregated_metrics["total_conversations"] = total_convs
        self.aggregated_metrics["total_leads_captured"] = successful_leads
        self.aggregated_metrics["conversion_rate"] = (
            successful_leads / total_convs * 100 if total_convs > 0 else 0
        )


#### SECTION 5: A/B TESTING FRAMEWORK

In [16]:
@dataclass
class Variant:
    """A/B test variant"""
    variant_id: str
    name: str
    config: Dict[str, Any]
    traffic_percentage: float = 50.0
    
    total_users: int = 0
    total_conversions: int = 0
    average_response_time: float = 0.0
    
    def conversion_rate(self) -> float:
        """Calculate conversion rate"""
        return (self.total_conversions / self.total_users * 100
                if self.total_users > 0 else 0)


class ABTest:
    """A/B Test manager"""
    
    def __init__(self, test_id: str, name: str, hypothesis: str):
        self.test_id = test_id
        self.name = name
        self.hypothesis = hypothesis
        self.created_at = datetime.now()
        self.variants: Dict[str, Variant] = {}
    
    def add_variant(self, variant: Variant):
        """Add a variant to the test"""
        self.variants[variant.variant_id] = variant
    
    def assign_variant(self, user_id: str) -> Variant:
        """Assign user to variant based on traffic split"""
        user_hash = hash(user_id) % 100
        
        total_traffic = 0
        for variant in self.variants.values():
            total_traffic += variant.traffic_percentage
            if user_hash < total_traffic:
                return variant
        
        return next(iter(self.variants.values()))
    
    def record_user(self, variant_id: str):
        """Record user interaction with variant"""
        if variant_id in self.variants:
            self.variants[variant_id].total_users += 1
    
    def record_conversion(self, variant_id: str):
        """Record conversion for variant"""
        if variant_id in self.variants:
            self.variants[variant_id].total_conversions += 1
    
    def get_results(self) -> Dict[str, Any]:
        """Get test results"""
        variants_data = {}
        for variant_id, variant in self.variants.items():
            variants_data[variant_id] = {
                "name": variant.name,
                "total_users": variant.total_users,
                "total_conversions": variant.total_conversions,
                "conversion_rate": variant.conversion_rate(),
            }
        
        return {
            "test_id": self.test_id,
            "name": self.name,
            "hypothesis": self.hypothesis,
            "created_at": self.created_at.isoformat(),
            "variants": variants_data,
        }


class ABTestingFramework:
    """Central A/B testing framework"""
    
    def __init__(self):
        self.active_tests: Dict[str, ABTest] = {}
        self.completed_tests: List[ABTest] = []
    
    def create_test(self, test_id: str, name: str, hypothesis: str) -> ABTest:
        """Create new A/B test"""
        test = ABTest(test_id, name, hypothesis)
        self.active_tests[test_id] = test
        return test
    
    def get_test(self, test_id: str) -> ABTest:
        """Get test by ID"""
        return self.active_tests.get(test_id)
    
    def conclude_test(self, test_id: str, winner_id: str = None):
        """Conclude a test"""
        if test_id in self.active_tests:
            test = self.active_tests[test_id]
            self.completed_tests.append(test)
            del self.active_tests[test_id]

#### SECTION 6: MAIN AGENT (LANGGRAPH)

In [19]:
!pip install langchain-anthropic

In [ ]:
from typing import Annotated, TypedDict
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages


class AgentState(TypedDict):
    """Define agent state"""
    messages: Annotated[list, add_messages]
    lead_data: LeadData
    intent: str
    kb_context: str
    last_action: str


class AutoStreamAgent:
    """Production-grade conversational agent for AutoStream"""
    
    def __init__(self, model_name: str = "claude-3-5-sonnet-20241022"):
        """Initialize agent"""
        self.llm = ChatAnthropic(model=model_name, temperature=0.7)
        self.kb = KnowledgeBase()
        self.graph = self._build_graph()
        self.compiled_graph = self.graph.compile()
    
    def _build_graph(self):
        """Construct the LangGraph workflow"""
        workflow = StateGraph(AgentState)
        
        # Add nodes
        workflow.add_node("intent_classification", self._classify_intent)
        workflow.add_node("retrieve_knowledge", self._retrieve_knowledge)
        workflow.add_node("respond", self._generate_response)
        workflow.add_node("lead_qualification", self._qualify_lead)
        workflow.add_node("collect_lead_info", self._collect_lead_info)
        workflow.add_node("capture_lead", self._capture_lead)
        
        # Add edges
        workflow.add_edge(START, "intent_classification")
        workflow.add_conditional_edges(
            "intent_classification",
            self._route_by_intent,
            {
                "greeting": "respond",
                "inquiry": "retrieve_knowledge",
                "high_intent": "lead_qualification"
            }
        )
        
        workflow.add_edge("retrieve_knowledge", "respond")
        workflow.add_edge("respond", END)
        workflow.add_edge("lead_qualification", "collect_lead_info")
        
        workflow.add_conditional_edges(
            "collect_lead_info",
            self._check_lead_complete,
            {
                "complete": "capture_lead",
                "incomplete": "respond"
            }
        )
        
        workflow.add_edge("capture_lead", END)
        
        return workflow
    
    def _classify_intent(self, state: AgentState) -> dict:
        """Classify user intent"""
        last_message = state["messages"][-1].content if state["messages"] else ""
        
        if LeadQualifier.is_high_intent(last_message):
            intent = "high_intent"
        elif any(word in last_message.lower() for word in ["hi", "hello", "hey", "thanks"]):
            intent = "greeting"
        else:
            intent = "inquiry"
        
        return {"intent": intent, "last_action": "classified_intent"}
    
    def _route_by_intent(self, state: AgentState) -> str:
        """Route conversation based on intent"""
        return state["intent"]
    
    def _retrieve_knowledge(self, state: AgentState) -> dict:
        """Retrieve knowledge from RAG"""
        last_message = state["messages"][-1].content if state["messages"] else ""
        retrieval_result = self.kb.retrieve(last_message)
        context = "\n".join(retrieval_result["context"])
        
        return {
            "kb_context": context,
            "last_action": "retrieved_knowledge"
        }
    
    def _generate_response(self, state: AgentState) -> dict:
        """Generate response using LLM"""
        system_prompt = """You are AutoStream's friendly AI assistant. You help content creators 
understand our video editing platform and guide them toward signing up.
Keep responses concise and helpful."""
        
        messages = state["messages"].copy()
        
        if state.get("kb_context"):
            context_msg = f"\n\n[Knowledge Base Context]\n{state['kb_context']}"
            if messages and isinstance(messages[-1], HumanMessage):
                messages[-1] = HumanMessage(
                    content=messages[-1].content + context_msg
                )
        
        response = self.llm.invoke(
            [SystemMessage(content=system_prompt)] + messages
        )
        
        return {
            "messages": [response],
            "last_action": "generated_response"
        }
    
    def _qualify_lead(self, state: AgentState) -> dict:
        """Qualify lead"""
        response = AIMessage(content="Great! To get started, what's your name?")
        return {
            "messages": [response],
            "last_action": "qualified_lead"
        }
    
    def _collect_lead_info(self, state: AgentState) -> dict:
        """Collect lead information"""
        lead_data = state.get("lead_data") or LeadData()
        last_message = None
        
        for msg in reversed(state["messages"]):
            if isinstance(msg, HumanMessage):
                last_message = msg.content
                break
        
        if last_message:
            lead_data = self._extract_user_info(last_message, lead_data)
        
        # Generate prompt for missing info
        missing_fields = []
        if not lead_data.name:
            missing_fields.append("their name")
        if not lead_data.email:
            missing_fields.append("their email")
        if not lead_data.platform:
            missing_fields.append("their content platform")
        
        if missing_fields:
            prompt = f"Ask for {missing_fields[0]}."
        else:
            prompt = "All info collected"
        
        response = AIMessage(content=f"Got it! Now please provide {prompt}")
        
        return {
            "messages": [response],
            "lead_data": lead_data,
            "last_action": "collecting_lead_info"
        }
    
    def _extract_user_info(self, message: str, lead_data: LeadData) -> LeadData:
        """Extract user info from message"""
        # Extract email
        email_pattern = r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b'
        emails = re.findall(email_pattern, message)
        if emails and validate_email(emails[0]):
            lead_data.email = emails[0]
        
        # Extract platform
        platforms = ["youtube", "instagram", "tiktok", "linkedin", "twitch", "facebook"]
        for platform in platforms:
            if platform in message.lower():
                lead_data.platform = platform.capitalize()
                break
        
        # Extract name
        words = message.split()
        for word in words:
            if word[0].isupper() and not any(char.isdigit() for char in word):
                if validate_name(word):
                    lead_data.name = word
                    break
        
        return lead_data
    
    def _check_lead_complete(self, state: AgentState) -> str:
        """Check if all lead info is collected"""
        lead_data = state.get("lead_data") or LeadData()
        return "complete" if lead_data.is_complete() else "incomplete"
    
    def _capture_lead(self, state: AgentState) -> dict:
        """Capture the lead"""
        lead_data = state.get("lead_data") or LeadData()
        
        if not lead_data.is_complete():
            return {"last_action": "lead_capture_failed"}
        
        result = mock_lead_capture(
            name=lead_data.name,
            email=lead_data.email,
            platform=lead_data.platform
        )
        
        success_msg = AIMessage(content=f"""
Great! I've captured your information:
- **Name:** {lead_data.name}
- **Email:** {lead_data.email}
- **Platform:** {lead_data.platform}

Our team will contact you within 24 hours! 🎉
""")
        
        return {
            "messages": [success_msg],
            "lead_data": lead_data,
            "last_action": "lead_captured"
        }
    
    def chat(self, user_message: str) -> str:
        """Run one conversation turn"""
        initial_state = {
            "messages": [HumanMessage(content=user_message)],
            "lead_data": LeadData(),
            "intent": "",
            "kb_context": "",
            "last_action": ""
        }
        
        result = self.compiled_graph.invoke(initial_state)
        
        for msg in reversed(result["messages"]):
            if isinstance(msg, AIMessage):
                return msg.content
        
        return "I'm here to help. What would you like to know about AutoStream?"
    
    def run_conversation(self) -> None:
        """Run interactive conversation loop"""
        print("🤖 AutoStream AI Agent")
        print("=" * 50)
        print("Hi! I'm here to help you learn about AutoStream.")
        print("Type 'exit' to quit.\n")
        
        while True:
            user_input = input("You: ").strip()
            
            if user_input.lower() in ["exit", "quit"]:
                print("\nThank you for chatting with AutoStream! 👋")
                break
            
            if not user_input:
                continue
            
            response = self.chat(user_input)
            print(f"\nAgent: {response}\n")


#### HELPER FUNCTIONS & EXECUTION

In [22]:
class AutoStreamAgent:
    def __init__(self):
        self.name = "AutoStream AI"

    def chat(self, message):
        # simple demo response
        return f"I received: {message}"

def run_demo_conversation():
    """Run demo conversation"""
    print("\n" + "="*60)
    print("🤖 AutoStream AI Agent - Demo")
    print("="*60 + "\n")
    
    agent = AutoStreamAgent()
    
    demo_messages = [
        "Hi, what's your pricing?",
        "That sounds good! I want the Pro plan for my YouTube channel.",
        "My name is Sarah Chen",
        "sarah.chen@gmail.com",
        "YouTube"
    ]
    
    for msg in demo_messages:
        print(f"You: {msg}")
        response = agent.chat(msg)
        print(f"Agent: {response}\n")


if __name__ == "__main__":
    # Example usage
    agent = AutoStreamAgent()
    
    # Interactive mode
    # agent.run_conversation()
    
    # Demo mode
    run_demo_conversation()



🤖 AutoStream AI Agent - Demo

You: Hi, what's your pricing?
Agent: I received: Hi, what's your pricing?

You: That sounds good! I want the Pro plan for my YouTube channel.
Agent: I received: That sounds good! I want the Pro plan for my YouTube channel.

You: My name is Sarah Chen
Agent: I received: My name is Sarah Chen

You: sarah.chen@gmail.com
Agent: I received: sarah.chen@gmail.com

You: YouTube
Agent: I received: YouTube

